# Experimento — MLP, fase 5: tudo no placar comum

Até a fase 4, cada relatório usou uma régua diferente: o teste na fase 2, uma validação contaminada pela parada na fase 3 e um k-fold com intervalo estreito demais na fase 4. Por isso ficava difícil dizer se alguma coisa tinha melhorado.

Esta fase mede tudo no **placar comum** (`comum.py`, descrito em `pre-proposta.md`), a mesma régua que os outros modelos do grupo vão usar:

- **k-fold estratificado repetido, 3 × 5**, nas sementes 101, 102 e 103, sobre os 75% de treino. O teste não entra.
- Em cada dobra, **80% do treino ajustam e 20% decidem a parada**. Depois a rede retreina no treino inteiro da dobra. A dobra de validação só mede.
- Toda comparação é **pareada nos mesmos clientes**, com IC95 por bootstrap de clientes, e dá uma de três palavras: **ganhou**, **empatou** ou **perdeu**. Só "ganhou" permite dizer "melhorou".

## Perguntas

1. **Alguma fase melhorou de verdade?** O trial 0 (1x10 padrão) e os campeões das fases 2, 3 e 4, todos na mesma régua.
2. **A rede pequena ganha só porque as grandes param cedo?** Seis arquiteturas, de 1x10 a 2x64, cada uma em três ritmos de treino: lr 1e-3 com paciência 10, lr 3e-4 com paciência 30 e lr 1e-4 com paciência 50.
3. **O Optuna acha algo melhor com uma medida menos ruidosa?** Na fase 4, cada trial valia 2 divisões e o vencedor foi sorte. Aqui cada trial vale **6 divisões** (as da fase 4, de 201 a 206, nunca as do placar). O espaço vai até paciência 50 e lr 1e-4, e inclui logística e tanh, como pedem os slides.
4. **Teste, uma vez só.** Entram a configuração escolhida pelo placar e a referência, como conferência.

## Como executar no Kaggle

1. *File → Import Notebook* → `experimentos/mlp/mlp_fase5.ipynb`, com o dataset anexado.
2. *Add-ons → Secrets:* `GITHUB_TOKEN` e `WANDB_API_KEY`. **Nunca escreva os tokens neste notebook.**
3. *Settings:* Internet **On** e acelerador **GPU T4 x2**.
4. Rode com `MODO_TESTE = True`. Depois, com `False`, use *Save Version → Save & Run All (Commit)*.

# 0. Preparação do ambiente

In [ ]:
# ===== Configuração da execução (edite aqui) =====
REPO_URL = "https://github.com/diegoflyra/churn_project.git"
BRANCH = "diego/mlp-diagnostico"             # branch com o código
RESULTS_REPO_URL = REPO_URL      # onde espelhar os resultados ("" = não enviar ao GitHub)
RESULTS_BRANCH = "resultados"    # branch órfã só de resultados
RUN_NAME = ""                    # vazio = "mlp-fase5-AAAAMMDD-HHMM"; fixe um nome para retomar
RESUME_FROM = ""                 # "" | "github" | pasta com os resultados de uma execução anterior
CSV_PATH = "/kaggle/input/datasets/diegolyra/churn-ibm/telco_ibm.csv"                    # caminho do CSV; vazio = procurar no repositório e em /kaggle/input
FASE2 = "resultados/mlp-fase2-20261104-0505"   # a referência: o vencedor da fase 2
USAR_GPU = True                  # placar em pilha na(s) GPU(s); sem GPU, uma rede por vez em vários processos
DISPOSITIVO = "cpu"              # onde roda modelo.py (Optuna e final): uma rede pequena por vez rende mais na CPU
N_PROCESSOS = 0                  # 0 = todos os núcleos
N_TRIALS = 100                   # trials do Optuna (o N da proposta)
REPS_OPTUNA = 6                  # divisões de 5 dobras por trial (a fase 4 usou 2, e o vencedor foi sorte)
N_SEMENTES = 5                   # sementes da avaliação final
USAR_WANDB = True                # False = não envia nada ao W&B, mesmo com a chave
USAR_GITHUB = True               # False = não espelha os resultados no GitHub, mesmo com o token
MODO_TESTE = False               # True = passada rápida para validar o notebook; os números não servem como resultado
HORAS_LIMITE = 10.5              # depois disso não começa etapas novas e encerra limpo (o Kaggle corta em 12 h)

In [ ]:
import importlib.util, os, shutil, subprocess, sys, time
from pathlib import Path

INICIO = time.time()


def _load_dotenv(path):
    """Tokens em um .env local (GITHUB_TOKEN, WANDB_API_KEY); nunca sobrescreve variáveis já definidas."""
    if os.path.isfile(path):
        for line in open(path, encoding="utf-8"):
            key, sep, value = line.strip().partition("=")
            if sep and not key.startswith("#") and value.strip() and not os.environ.get(key.strip()):
                os.environ[key.strip()] = value.strip().strip("'").strip('"')


def _secret(name):
    value = os.environ.get(name, "").strip()
    if value:
        return value
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        pass
    return ""


def _is_repo(p):
    return (p / "comum.py").exists() and (p / "experimentos" / "mlp" / "modelo.py").exists()


local = next((p for p in [Path.cwd(), *Path.cwd().parents] if _is_repo(p)), None)
if local:
    REPO_DIR = local
    _load_dotenv(REPO_DIR / ".env")
    GITHUB_TOKEN = _secret("GITHUB_TOKEN")
    print(f"Código: cópia local em {REPO_DIR}")
else:
    GITHUB_TOKEN = _secret("GITHUB_TOKEN")
    REPO_DIR = Path("/tmp/churn_project")
    url = REPO_URL.replace("https://", f"https://x-access-token:{GITHUB_TOKEN}@", 1) if GITHUB_TOKEN else REPO_URL
    print("GitHub: usando GITHUB_TOKEN" if GITHUB_TOKEN else "GitHub: clone sem token")
    shutil.rmtree(REPO_DIR, ignore_errors=True)
    if subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, url, str(REPO_DIR)]).returncode != 0:
        raise RuntimeError(f"Falha ao clonar a branch {BRANCH}. Confira se ela foi publicada; repositório privado exige GITHUB_TOKEN.")
    subprocess.run(["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", REPO_URL])  # o token não fica salvo no clone
    print(f"Código: {REPO_DIR}")
os.chdir(REPO_DIR)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout.strip())

# Instala só o que falta: o Kaggle já traz versões compatíveis com a GPU dele
IMPORTS = {"imbalanced-learn": "imblearn", "scikit-learn": "sklearn"}
faltam = [p.strip() for p in open("requirements.txt", encoding="utf-8")
          if p.strip() and importlib.util.find_spec(IMPORTS.get(p.strip(), p.strip())) is None]
if faltam:
    print("Instalando:", ", ".join(faltam))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltam])

In [ ]:
import json

sys.path[:0] = [str(REPO_DIR), str(REPO_DIR / "experimentos" / "mlp")]
os.environ["PYTHONPATH"] = os.pathsep.join([str(REPO_DIR), str(REPO_DIR / "experimentos" / "mlp"), os.environ.get("PYTHONPATH", "")])  # processos paralelos acham os módulos
os.environ.setdefault("TORCH_DISABLE_NATIVE_JIT", "1")
import fluxo

WORKDIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content" if os.path.isdir("/content") else str(REPO_DIR)
RUN_NAME = RUN_NAME or time.strftime("mlp-fase5-%Y%m%d-%H%M")
if MODO_TESTE and not RUN_NAME.startswith("teste-"):
    RUN_NAME = "teste-" + RUN_NAME
# O teste nunca se mistura com a execução real, e rodar localmente não sobrescreve o que está versionado em resultados/
PASTA = os.path.join(WORKDIR, "execucoes", RUN_NAME) if WORKDIR == str(REPO_DIR) else os.path.join(WORKDIR, "resultados_" + RUN_NAME)
os.makedirs(PASTA, exist_ok=True)
os.environ["CHURN_RESULTADOS"] = PASTA
os.environ["CHURN_CSV"] = fluxo.achar_csv(CSV_PATH)
if DISPOSITIVO:
    os.environ["CHURN_DEVICE"] = DISPOSITIVO
N_PROCESSOS = N_PROCESSOS or os.cpu_count() or 1
if MODO_TESTE:
    os.environ["CHURN_MAX_EPOCAS"] = "3"
    N_TRIALS, REPS_OPTUNA, N_SEMENTES = 4, 2, 2
    print("MODO_TESTE: 3 épocas e poucos trials. Os números não servem como resultado.")

if not USAR_GITHUB:
    RESULTS_REPO_URL = ""
if RESUME_FROM == "github":
    fluxo.baixar_github(PASTA, RUN_NAME, RESULTS_REPO_URL, GITHUB_TOKEN, RESULTS_BRANCH)
elif RESUME_FROM:
    shutil.copytree(RESUME_FROM, PASTA, dirs_exist_ok=True)


def salvar():
    return fluxo.salvar_github(PASTA, RUN_NAME, RESULTS_REPO_URL, GITHUB_TOKEN, RESULTS_BRANCH)


def salvar_json(nome, obj):
    json.dump(obj, open(os.path.join(PASTA, nome), "w"), indent=1, default=float)
    salvar()


_ultimo_backup = [time.time()]


def backup_periodico():
    if time.time() - _ultimo_backup[0] > 30 * 60:
        salvar(); _ultimo_backup[0] = time.time()


def continuar():
    return HORAS_LIMITE * 3600 - (time.time() - INICIO) > 0


import matplotlib.pyplot as plt, numpy as np, pandas as pd
from scipy.stats import ks_2samp
import comum, modelo as M, blocos as BL, fase3 as F3, fase4 as F4, fase5 as F5, pilha as P
import torch

GPUS = P.dispositivos_disponiveis() if USAR_GPU and torch.cuda.is_available() else None
F5.usar_pilha(GPUS)
pd.set_option("display.max_columns", None, "display.width", 220, "display.max_colwidth", 70, "display.max_rows", 80)
print(f"RUN_NAME = {RUN_NAME} | resultados em {PASTA}")
print(f"CSV: {os.environ['CHURN_CSV']} | pilha: {GPUS or 'desligada'} | modelo.py: {M.DEVICE} | processos: {N_PROCESSOS}")

# O envio ao GitHub falhou a fase 3 inteira sem ninguém perceber: aqui ele é testado antes de qualquer treino
json.dump(dict(run=RUN_NAME, inicio=time.strftime("%Y-%m-%d %H:%M")), open(os.path.join(PASTA, "inicio.json"), "w"))
GITHUB_OK = bool(RESULTS_REPO_URL and GITHUB_TOKEN) and salvar()
if RESULTS_REPO_URL and not GITHUB_OK:
    print("=" * 90 + "\nATENÇÃO: o envio ao GitHub falhou. Troque o GITHUB_TOKEN nos Secrets ou baixe a pasta de resultados no fim.\n" + "=" * 90)
else:
    print("GitHub:", "envio testado e funcionando" if GITHUB_OK else "desligado")

wandb_run = None
wandb_key = _secret("WANDB_API_KEY") if USAR_WANDB else ""
if wandb_key or os.environ.get("WANDB_MODE") == "offline":
    import wandb
    if wandb_key:
        os.environ["WANDB_API_KEY"] = wandb_key
    wandb_run = wandb.init(project="churn-if702", entity=_secret("WANDB_ENTITY") or None, name=RUN_NAME, dir=WORKDIR,
                           group="mlp", config=dict(modelo="mlp", fase=5, processos=N_PROCESSOS, gpus=GPUS, teste=MODO_TESTE))
else:
    print("W&B desativado (sem chave ou USAR_WANDB = False).")


def logar(**k):
    if wandb_run is not None:
        wandb_run.log(k)


REFERENCIA = json.load(open(os.path.join(REPO_DIR, FASE2, "blocos", "b12_arquitetura.json")))["campeao"]
REFERENCIA = dict(hp={**REFERENCIA["hp"], "plano": "N"}, dados=REFERENCIA["dados"], extras=REFERENCIA["extras"])
# Os campeões das fases 3 e 4, escritos aqui para o notebook não depender das pastas de resultados delas
FASE3_VAL = {'hp': {'camadas': 3, 'unidades': 16, 'ativ': 'gelu', 'otim': 'adam', 'lr': 0.001, 'dropout': 0.2, 'l2': 1e-06, 'perda': 'ce', 'paciencia': 10, 'plano': 'DND', 'parada': 'ks'}, 'dados': {}, 'extras': ['cliente_novo', 'delta_preco']}
FASE3_TESTE = {'hp': {'camadas': 2, 'unidades': 47, 'ativ': 'relu', 'otim': 'adam', 'lr': 0.0001564570789442291, 'dropout': 0.3215897831163661, 'l2': 1.1107958156042701e-06, 'perda': 'mse', 'paciencia': 10, 'plano': 'DD', 'peso_pos': 1.5, 'funil': True, 'suav': 0.0898753403823129, 'pen_logit': 0.0, 'lote': 512, 'parada': 'perda'}, 'dados': {'over': False}, 'extras': ['cliente_novo', 'delta_preco']}
FASE4 = {'hp': {'camadas': 1, 'unidades': 10, 'ativ': 'relu', 'otim': 'adam', 'lr': 0.001, 'dropout': 0.0, 'l2': 1e-06, 'perda': 'ce', 'paciencia': 10, 'plano': 'N'}, 'dados': {}, 'extras': ['cliente_novo', 'delta_preco', 'tenure_1']}
print("Referência (vencedor da fase 2):", json.dumps(REFERENCIA, ensure_ascii=False))

In [ ]:
# Verificações rápidas antes de gastar tempo de treino
assert comum.assinatura_teste() == "e90ba6fab761f037", "o teste mudou: os números não seriam comparáveis com os outros modelos"
rng = np.random.default_rng(1); y_ = (rng.random(2000) < .265).astype(int); p_ = np.clip(rng.normal(.3 + .25 * y_, .2), 0, 1)
assert abs(comum.ks(y_, p_) - ks_2samp(p_[y_ == 1], p_[y_ == 0]).statistic) < 1e-9
D = F5.divs()
assert len(D) == 15
for d in D:
    assert not set(d["treino"]) & set(d["validacao"]) and not set(d["ajuste"]) & set(d["parada"])
y_treino = F4._dados()[2].values.astype(int)
print(f"teste com a assinatura oficial | KS conferido contra scipy | 15 divisões do placar sem sobreposição | assinatura do treino {F5.assinatura()}")


def mostrar(linhas, campos=("vs_trial0", "vs_referencia")):
    display(F5.tabela(linhas, campos).round(4))
    salvar()

# 1. Alguma fase melhorou de verdade?

Todas as versões na mesma régua. `ks_kfold` é a média dos 15 KS das dobras. `ks_oof` é o KS da previsão fora da dobra de cada cliente, com a média das 3 sementes. É nela que o veredito é calculado.

- `vs_trial0`: contra a 1x10 padrão, sem nenhuma feature nova.
- `vs_referencia`: contra o campeão da fase 2.

In [ ]:
REF = ("mlp", "referencia")
T0 = ("mlp", "trial0")
FASES = [("trial0", F5.TRIAL0, None),
         ("referencia", REFERENCIA, {"vs_trial0": T0}),
         ("fase3_val", FASE3_VAL, {"vs_trial0": T0, "vs_referencia": REF}),
         ("fase3_teste", FASE3_TESTE, {"vs_trial0": T0, "vs_referencia": REF}),
         ("fase4", FASE4, {"vs_trial0": T0, "vs_referencia": REF})]
t0 = time.time()
linhas_fases = F5.garantir(PASTA, FASES, n_jobs=N_PROCESSOS)
print(f"{(time.time() - t0) / 60:.1f} min")
mostrar(linhas_fases)
fase4_ganhou = (linhas_fases[-1].get("vs_referencia") or {}).get("veredito") == "ganhou"
BASE = FASE4 if fase4_ganhou else REFERENCIA
print("Base das próximas etapas:", "fase 4 (tenure_1 ganhou no placar)" if fase4_ganhou else "referência (tenure_1 não ganhou no placar)", BASE["extras"])
logar(etapa="fases", ks_referencia=linhas_fases[1]["ks_oof"])

# 2. A rede pequena ganha só porque as grandes param cedo?

Cada arquitetura em três ritmos de treino. Com lr menor e paciência maior, uma rede grande tem mais épocas antes de começar a decorar. Se a 1x10 só vencia por causa do lr de 1e-3, alguma rede grande deve **ganhar** da referência no ritmo lento. As redes com mais de 10 unidades usam Dropout, nunca BN, que perdeu em todas as camadas na fase 3.

In [ ]:
GRADE = F5.grade_arquiteturas(BASE)
linhas_grade = []
if continuar():
    t0 = time.time()
    linhas_grade = F5.garantir(PASTA, [(r, c, {"vs_trial0": T0, "vs_referencia": REF}) for r, c in GRADE.items()], n_jobs=N_PROCESSOS)
    print(f"{(time.time() - t0) / 60:.1f} min")
    tg = F5.tabela(linhas_grade)
    tg["arquitetura"] = tg.rotulo.str.split("_").str[1]
    tg["ritmo"] = tg.rotulo.str.split("_", n=2).str[2]
    ordem = list(F5.RITMOS)
    print("KS da OOF (veredito contra a referência abaixo):")
    display(tg.pivot(index="arquitetura", columns="ritmo", values="ks_oof")[ordem].loc[list(F5.ARQUITETURAS)].round(4))
    display(tg.pivot(index="arquitetura", columns="ritmo", values="vs_referencia")[ordem].loc[list(F5.ARQUITETURAS)])
    print("Épocas medianas da parada:")
    display(tg.pivot(index="arquitetura", columns="ritmo", values="epocas")[ordem].loc[list(F5.ARQUITETURAS)])
    print("MSE no treino x validação (distância grande = decorou):")
    display(tg.set_index("rotulo")[["mse_treino", "mse_val"]].round(4))
    salvar()

# 3. Optuna com 6 divisões por trial

Cada trial vale o KS médio do retreino em 6 divisões de 5 dobras (201 a 206). A medida é a da fase 4, mas com divisões que nunca são as do placar. O trial 0 é a base. O vencedor só conta se **ganhar no placar**, em divisões que a busca não viu.

In [ ]:
st, melhor_optuna = None, None
if continuar():
    t0 = time.time()
    st, melhor_optuna = F5.estudar(PASTA, BASE, N_TRIALS, reps=tuple(range(201, 201 + REPS_OPTUNA)), n_jobs=N_PROCESSOS,
                                   continuar=continuar, ao_salvar=backup_periodico)
    print(f"{(time.time() - t0) / 60:.1f} min | trial 0 {st.trials[0].value:.4f} | melhor {st.best_value:.4f}")
    print(json.dumps(melhor_optuna, ensure_ascii=False))
    resumo = F5.resumo_optuna(st)
    for k in ("camadas", "unidades", "ativ", "lr", "paciencia", "perda", "otim"):
        if k in resumo:
            print(f"  {k:<10}", {a: f"{b['max']:.4f} (n={int(b['count'])})" for a, b in resumo[k].items()})
    salvar_json("mlp_fase5_optuna_resumo.json", dict(trial0=st.trials[0].value, melhor=st.best_value, melhor_config=melhor_optuna, por_parametro=resumo))
    if melhor_optuna != BASE:
        mostrar(F5.garantir(PASTA, [("optuna", melhor_optuna, {"vs_trial0": T0, "vs_referencia": REF})], n_jobs=N_PROCESSOS))

# 4. Escolha pelo placar

Só concorre quem **ganhou** da referência. Entre esses, fica o maior KS da OOF. Se ninguém ganhou, a referência continua: no empate, fica a rede menor.

In [ ]:
todas = [r for r in comum.ler_placar(PASTA) if r["modelo"] == "mlp"]
escolhida = F5.escolher(todas)
ROT = escolhida["rotulo"] if escolhida else "referencia"
CFG = escolhida["config"] if escolhida else REFERENCIA
print("Escolhida:", ROT, "| ganhou da referência no placar" if escolhida else "| ninguém ganhou da referência: ela continua")
print(json.dumps(CFG, ensure_ascii=False))
ganharam = [r["rotulo"] for r in todas if (r.get("vs_referencia") or {}).get("veredito") == "ganhou"]
perderam = [r["rotulo"] for r in todas if (r.get("vs_referencia") or {}).get("veredito") == "perdeu"]
print(f"Ganharam da referência: {ganharam or 'nenhuma'}\nPerderam: {perderam or 'nenhuma'}")

# 5. Teste, uma vez só

Treino inteiro: 20% internos decidem as épocas e a rede retreina em tudo, com 5 sementes. O limiar das métricas de classificação vem da OOF do placar. **O teste confere, mas não decide:** se contradisser o placar, os dois números ficam registrados e a escolha não muda.

In [ ]:
finais = {}
arq_final = os.path.join(PASTA, "mlp_fase5_final.json")
if os.path.isfile(arq_final):
    finais = json.load(open(arq_final)); print("Teste já feito nesta execução: reaproveitado, sem treinar de novo.")
    for rot, f in finais.items():
        if rot != "delta_teste":
            m = f["metricas"]; print(f"{rot:<12} KS {m['ks']:.4f} IC95 {np.round(m['ks_ic95'], 4)} | AUROC {m['auroc']:.4f}")
elif continuar():
    for rot, c in (("referencia", REFERENCIA), (ROT, CFG)):
        if rot in finais:
            continue
        finais[rot] = F5.final(c, N_SEMENTES)
        m = F5.metricas_teste(PASTA, rot, finais[rot])
        finais[rot]["metricas"] = m
        comum.registrar_teste_placar(PASTA, "mlp", rot, finais[rot]["y_teste"], finais[rot]["p_sementes"])
        print(f"{rot:<12} KS {m['ks']:.4f} IC95 {np.round(m['ks_ic95'], 4)} | AUROC {m['auroc']:.4f} | MSE {m['mse']:.4f} | F1 {m['f1']:.4f} | por semente {np.round(m['ks_por_semente'], 4)}")
    if ROT != "referencia":
        dt = F5.delta_teste(finais["referencia"], finais[ROT])
        print(f"{ROT} - referência no teste: {dt['delta']:+.4f} IC95 [{dt['ic95'][0]:+.4f}, {dt['ic95'][1]:+.4f}] (conferência)")
        finais["delta_teste"] = dt
    salvar_json("mlp_fase5_final.json", finais)

In [ ]:
# O placar desta execução: uma linha por versão, com o teste só onde ele foi usado
tab = comum.tabela_placar(PASTA)
display(tab.round(4))
tab.to_csv(os.path.join(PASTA, "placar_mlp.csv"), index=False)
salvar_json("mlp_fase5_campeao.json", dict(rotulo=ROT, config=CFG))
if wandb_run is not None:
    wandb_run.summary.update(dict(escolhida=ROT, config=json.dumps(CFG)))
    wandb_run.finish()
print(f"Tempo total: {(time.time() - INICIO) / 3600:.2f} h")
print("Resultados em", PASTA)
print(sorted(os.listdir(PASTA)))

### 📝 Análise da fase 5

- **Alguma fase melhorou na régua comum?** _…_
- **Rede maior com treino mais lento ganhou?** _…_
- **O Optuna com 6 divisões achou algo que ganhou no placar?** _…_
- **O teste confirmou ou contradisse?** _…_
- **Próxima hipótese:** _…_